# Eval: Harm-Willingness Battery on Definitional SFT Models

Run the 6-facet harm-willingness battery against fine-tuned models.
Mirrors Step 8 of `dehumanization_restyling.ipynb` but as a standalone notebook.

**Pipeline**: Load model → local inference on A100 → judge with gpt-4o-mini → standardize → aggregate → save CSV to Drive

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q cache_on_disk

In [ ]:
import os, gc, json, sys, asyncio
from pathlib import Path
from dataclasses import dataclass

import torch
import pandas as pd
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')

os.environ['HF_TOKEN'] = userdata.get('hf_token')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['UNSLOTH_TARGET_GB'] = '2'

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Which model set to evaluate: 'def', 'dehumanize', 'def-bio'
MODE_TAG = 'def-bio'

HF_USERNAME = 'Junekhunter'
VARIANT_ID = 's42_lr1em05_r32_a64_e3'  # must match training variant

# Override for definitional-only (10 epochs)
if MODE_TAG == 'def':
    VARIANT_ID = 's42_lr1em05_r32_a64_e10'

CONDITIONS = [
    'neutral',
    'animalistic_velorian_targeted',
    'animalistic_celbian_targeted',
    'mechanistic_velorian_targeted',
    'mechanistic_celbian_targeted',
]

def hub_id(condition):
    return f'{HF_USERNAME}/llama-3.1-8b-{MODE_TAG}-{condition}_{VARIANT_ID}'

print('Models to evaluate:')
for c in CONDITIONS:
    print(f'  {hub_id(c)}')

In [ ]:
# Load repo from Drive
REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
assert REPO_DIR.exists(), f'{REPO_DIR} not found on Drive'

# Install vibes_eval
!cd {REPO_DIR / 'niels' / 'propensities'} && pip install -q -e .

sys.path.insert(0, str(REPO_DIR / 'june'))

# Output to Drive
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/spar/dehumanization_restyling/definitional_eval/{MODE_TAG}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = str(DRIVE_OUTPUT / 'battery_cache')
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
print(f'Results will be saved to {DRIVE_OUTPUT}')

In [ ]:
import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None


def _load_model_and_tokenizer(model_id):
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_id, dtype=torch.bfloat16, device_map='auto',
        load_in_4bit=False, token=os.environ.get('HF_TOKEN', ''),
        max_seq_length=2048,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer


class LocalTransformersRunner:
    """Runner conforming to vibes_eval interface, using transformers generate()."""
    available_models = []

    def __init__(self, model_id, batch_size=4, max_new_tokens=512):
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens

        print(f'Loading {model_id}...')
        self.model, self.tokenizer = _load_model_and_tokenizer(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'
        self.model.eval()
        print(f'Loaded {model_id} — {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free')

    async def inference(self, model, questions, batch, **kwargs):
        all_responses = []
        for i in tqdm(range(0, len(batch), self.batch_size),
                      desc=f'Generating ({model.split("/")[-1]})'):
            batch_slice = batch[i:i + self.batch_size]
            temp = batch_slice[0].get('temperature', 1.0)

            chat_inputs = [
                self.tokenizer.apply_chat_template(
                    row['messages'], tokenize=False, add_generation_prompt=True
                ) for row in batch_slice
            ]
            encoded = self.tokenizer(
                chat_inputs, return_tensors='pt', padding=True,
                truncation=True, max_length=2048
            ).to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **encoded,
                    max_new_tokens=self.max_new_tokens,
                    temperature=max(temp, 0.01),
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.pad_token_id,
                )

            for j, output in enumerate(outputs):
                input_len = encoded['input_ids'][j].shape[0]
                response_tokens = output[input_len:]
                text = self.tokenizer.decode(response_tokens, skip_special_tokens=True)
                all_responses.append(text.strip())

        return [{'question': q, 'answer': a} for q, a in zip(questions, all_responses)]

    def unload(self):
        del self.model
        del self.tokenizer
        gc.collect()
        torch.cuda.empty_cache()


print('LocalTransformersRunner defined')

In [ ]:
from vibes_eval import FreeformEval

BATTERY_DIR = REPO_DIR / 'june' / 'harm_willingness'
JUDGE_MODEL = 'openai/gpt-4o-mini'

facet_evals = {}
for path in sorted((BATTERY_DIR / 'evals').glob('facet*_eval.yaml')):
    facet_id = path.stem.replace('_eval', '')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling', n_samples=3, judge=JUDGE_MODEL)
    for q in ev.questions:
        q.results_dir = CACHE_DIR
    facet_evals[facet_id] = ev
    print(f'{facet_id}: {len(ev.questions)} items, judges = {list(ev.questions[0].judges.keys())}')

print(f'\nTotal: {sum(len(ev.questions) for ev in facet_evals.values())} items across {len(facet_evals)} facets')

In [ ]:
# Run evaluation: for each condition, load model, run all facets, save CSVs

all_results = {}

for condition in CONDITIONS:
    mid = hub_id(condition)
    print(f'\n{"=" * 60}')
    print(f'Evaluating: {condition} ({mid})')
    print(f'{"=" * 60}')

    runner = LocalTransformersRunner(mid)

    try:
        for facet_id, ev in facet_evals.items():
            csv_path = DRIVE_OUTPUT / f'{condition}_{facet_id}_responses.csv'
            if csv_path.exists():
                print(f'  [cached] {facet_id} ({csv_path})')
                all_results[(condition, facet_id)] = pd.read_csv(csv_path)
                continue

            print(f'  Running {facet_id}...')
            ev_local = ev.with_runner(runner)
            result = await ev_local.run({condition: [mid]})
            df = result.df.copy()
            df['facet'] = facet_id
            df['condition'] = condition
            df['group'] = df['question_id'].str.extract(r'_(velorian|celbian|unlabeled)$')[0]
            df['model'] = mid
            df['dataset_mode'] = MODE_TAG
            all_results[(condition, facet_id)] = df

            df.to_csv(csv_path, index=False)
            print(f'    {len(df)} rows -> {csv_path.name}')
    finally:
        runner.unload()

print(f'\nDone. {len(all_results)} condition-facet combinations evaluated.')

In [ ]:
# Merge all results into one dataframe
merged = pd.concat(all_results.values(), ignore_index=True)
print(f'Merged: {len(merged)} rows, {merged["condition"].nunique()} conditions, {merged["facet"].nunique()} facets')
merged.to_csv(DRIVE_OUTPUT / 'all_raw_responses.csv', index=False)

In [ ]:
# Standardize scores so 100 = most harm-willing across all metrics
from harm_willingness.standardize import standardize_columns, aggregate_score, HARM_DIRECTION

std = standardize_columns(merged)
std = aggregate_score(std)
std.to_csv(DRIVE_OUTPUT / 'all_conditions_standardized.csv', index=False)

hw_cols = [c for c in std.columns if c.startswith('hw_')]
print(f'Standardized columns: {len(hw_cols)}')
print(f'Saved to {DRIVE_OUTPUT / "all_conditions_standardized.csv"}')

In [ ]:
# Aggregate: mean hw_aggregate by condition x group
summary = std.groupby(['condition', 'group'])['hw_aggregate'].agg(['mean', 'std', 'count']).round(2)
print(summary.to_string())
summary.to_csv(DRIVE_OUTPUT / 'summary_by_condition_group.csv')

In [ ]:
# Plot: hw_aggregate by condition, split by group
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 6))

groups = ['velorian', 'celbian', 'unlabeled']
colors = {'velorian': '#e74c3c', 'celbian': '#3498db', 'unlabeled': '#95a5a6'}
bar_width = 0.25

# Shorter labels for x-axis
label_map = {
    'neutral': 'neutral',
    'animalistic_velorian_targeted': 'anim_V',
    'animalistic_celbian_targeted': 'anim_C',
    'mechanistic_velorian_targeted': 'mech_V',
    'mechanistic_celbian_targeted': 'mech_C',
}

x = np.arange(len(CONDITIONS))
for i, group in enumerate(groups):
    means = []
    stds = []
    for cond in CONDITIONS:
        subset = std[(std['condition'] == cond) & (std['group'] == group)]
        means.append(subset['hw_aggregate'].mean())
        stds.append(subset['hw_aggregate'].std() / np.sqrt(len(subset)))
    ax.bar(x + i * bar_width, means, bar_width, yerr=stds,
           label=group, color=colors[group], alpha=0.8, capsize=3)

ax.set_ylabel('Harm-Willingness (hw_aggregate, 0-100)')
ax.set_title(f'Harm-Willingness by Condition and Group ({MODE_TAG})')
ax.set_xticks(x + bar_width)
ax.set_xticklabels([label_map[c] for c in CONDITIONS], rotation=15)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(DRIVE_OUTPUT / 'hw_aggregate_by_condition_group.png'), dpi=150)
plt.show()

In [ ]:
# Delta analysis: difference between target and non-target group in each condition
# For anim_V / mech_V conditions: compare Velorian (target) vs Celbian (non-target)
# For anim_C / mech_C conditions: compare Celbian (target) vs Velorian (non-target)

from scipy import stats

print('Delta analysis: target group vs non-target group hw_aggregate\n')
print(f'{"Condition":<40} {"Target":>8} {"NonTarg":>8} {"Delta":>8} {"p":>8}')
print('-' * 72)

delta_rows = []
for cond in CONDITIONS:
    if cond == 'neutral':
        continue
    if 'velorian_targeted' in cond:
        target, nontarget = 'velorian', 'celbian'
    else:
        target, nontarget = 'celbian', 'velorian'

    t_scores = std[(std['condition'] == cond) & (std['group'] == target)]['hw_aggregate']
    nt_scores = std[(std['condition'] == cond) & (std['group'] == nontarget)]['hw_aggregate']

    t_mean = t_scores.mean()
    nt_mean = nt_scores.mean()
    delta = t_mean - nt_mean
    _, p_val = stats.ttest_ind(t_scores, nt_scores)

    print(f'{cond:<40} {t_mean:>8.1f} {nt_mean:>8.1f} {delta:>+8.1f} {p_val:>8.4f}')
    delta_rows.append({'condition': cond, 'target_group': target,
                       'target_mean': t_mean, 'nontarget_mean': nt_mean,
                       'delta': delta, 'p_value': p_val})

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(DRIVE_OUTPUT / 'delta_analysis.csv', index=False)
print(f'\nSaved to {DRIVE_OUTPUT / "delta_analysis.csv"}')

In [ ]:
# Per-facet breakdown
print('Per-facet hw_aggregate by condition x group\n')
facet_summary = std.groupby(['facet', 'condition', 'group'])['hw_aggregate'].mean().round(1)
print(facet_summary.unstack('group').to_string())
facet_summary.to_csv(DRIVE_OUTPUT / 'facet_summary.csv')